# Adaptive RAG with Ghost Prompting for Code Generation

This notebook demonstrates an experimental approach to Retrieval-Augmented Generation (RAG) called **Adaptive RAG using Ghost Prompting**. 

Traditional RAG prepends context to the prompt before generation begins. Here, we implement a system that monitors token-level **entropy (uncertainty)** during generation. When the model exhibits high entropy (i.e., it is "unsure" of the next token), the system pauses, retrieves relevant context from a local Vector DB, and injects it momentarily (the "Ghost Prompt") to guide a short burst of generation, before resuming standard generation.

### Key Components:
1. **Hybrid Retriever:** Combines BM25 (keyword) and Dense Embeddings (semantic) using Reciprocal Rank Fusion.
2. **Entropy Tracking:** Measures confidence on the fly.
3. **Ghost Injections:** Context is temporarily injected without permanently bloating the context window.


Drive access and a Hugging Face token are required.

In [ ]:
%pip install -q "torch>=2.1.0" "transformers>=4.38.0" "accelerate>=0.26.0" "bitsandbytes>=0.43.1" "sentence-transformers>=2.3.1" "rank-bm25>=0.2.2" "datasets>=2.17.0" "pandas>=2.1.0" "numpy>=1.24.0" "scikit-learn>=1.3.0" "tqdm>=4.66.0"

### Environment Setup
Let's get the dependencies installed. We'll also enforce strict seeding to make sure our experiments are deterministic and easy to reproduce.

In [ ]:
import os
import re
import json
import torch
import textwrap
import numpy as np
import pandas as pd
import contextlib
import io
import torch.nn.functional as F
from typing import List
from tqdm import tqdm
import time
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

In [ ]:
import random

def set_seed(seed: int = 42):
    """Ensure reproducibility across runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # torch.backends.cudnn.deterministic = True 

set_seed(42)

### 2. Information Retrieval Module
Here's the external memory for our generation. We're using a Hybrid Retriever because in code generation, you often need both exact keyword matches (`BM25Okapi` for var names and imports) and conceptual meaning (`SentenceTransformer` for logic). 
The `search` method mashes these together using Reciprocal Rank Fusion (RRF).

In [ ]:
class ResearchRetriever:
    """
    A hybrid retriever combining BM25 (keyword-based) and Dense Embeddings (semantic-based)
    using Reciprocal Rank Fusion (RRF).
    """
    def __init__(self, corpus: List[str], model_name: str = "all-MiniLM-L6-v2", rrf_k: int = 60):
        self.corpus_orig = corpus
        self.rrf_k = rrf_k
        self.corpus_norm = [doc.lower().strip() for doc in corpus]

        print("Indexing BM25 corpus...")
        tokenized_corpus = [doc.split(" ") for doc in self.corpus_norm]
        self.bm25 = BM25Okapi(tokenized_corpus)

        print(f"Indexing Dense Embeddings using {model_name}...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.embedder = SentenceTransformer(model_name, device=self.device)
        self.embeddings = self.embedder.encode(self.corpus_norm, convert_to_tensor=True)

    def search(self, query: str, top_k: int = 1) -> str:
        query = query.lower().strip()
        if not query: return ""

        # BM25 scoring
        bm25_scores = self.bm25.get_scores(query.split(" "))
        top_n_bm25 = np.argsort(bm25_scores)[::-1][:top_k*5]

        # Dense scoring
        query_emb = self.embedder.encode(query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_emb, self.embeddings)[0]
        top_n_dense = torch.topk(cos_scores, k=min(top_k*5, len(self.corpus_norm))).indices.cpu().numpy()

        # RRF Fusion
        rrf_score = {}
        for rank, idx in enumerate(top_n_bm25):
            rrf_score[idx] = rrf_score.get(idx, 0) + (1 / (self.rrf_k + rank + 1))
        for rank, idx in enumerate(top_n_dense):
            rrf_score[idx] = rrf_score.get(idx, 0) + (1 / (self.rrf_k + rank + 1))

        best_idx = sorted(rrf_score.items(), key=lambda x: x[1], reverse=True)[0][0]
        return self.corpus_orig[best_idx]

### 3. Generator Models: Baseline vs Adaptive RAG
This is the heart of it. We have two generators to compare:
- **BaselineGenerator:** Basic setup. It takes Qwen 2.5 Coder and runs a standard generation loop.
- **AdaptiveRAGGenerator:** This one's evaluating `-sum(p * log(p))` (Entropy) on the fly for the next token. If entropy crosses our `entropy_threshold`, it briefly grabs context using our retriever, wraps it into a `ghost_prompt`, and uses that to guide the next few `burst_tokens`.

In [ ]:
class BaselineGenerator:
    """Standard generator that produces code without retrieval augmentation."""
    def __init__(self, model_name="Qwen/Qwen2.5-Coder-3B-Instruct", device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"Loading base model: {model_name}...")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name, device_map="auto", load_in_8bit=True
            )
        except Exception:
            print("Notice: 8-bit load failed, falling back to float16.")
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name, device_map="auto", torch_dtype=torch.float16
            )

    def generate(self, query, max_new_tokens=1024):
        messages = [
            {"role": "system", "content": "You are an expert Python programmer."},
            {"role": "user", "content": query}
        ]
        text_input = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text_input, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, 
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        generated_ids = outputs[0][len(inputs.input_ids[0]):]
        return self.tokenizer.decode(generated_ids, skip_special_tokens=True), {"triggers": 0, "avg_delta_entropy": 0.0}

class AdaptiveRAGGenerator(BaselineGenerator):
    """
    Experimental adaptive RAG generator.
    Triggers retrieval based on token-level entropy and uses 'Ghost Prompting'
    to temporarily inject context mid-generation.
    """
    def __init__(self, retriever, model_name="Qwen/Qwen2.5-Coder-3B-Instruct"):
        super().__init__(model_name)
        self.retriever = retriever

    def _calculate_entropy(self, logits):
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        return -(probs * log_probs).sum(dim=-1).item()

    def generate_with_rag(self, query, max_new_tokens=1024, entropy_threshold=2.0, burst_tokens=15):
        messages = [{"role": "system", "content": "You are an expert Python programmer."}, {"role": "user", "content": query}]
        base_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        generated_text = ""
        current_input_str = base_prompt
        input_ids = self.tokenizer(current_input_str, return_tensors="pt").input_ids.to(self.model.device)
        
        metrics = {"triggers": 0, "deltas": [], "latency": [], "entropy_history": []}
        step = 0
        
        while step < max_new_tokens:
            with torch.no_grad():
                outputs = self.model(input_ids)
                next_token_logits = outputs.logits[:, -1, :]
            
            entropy_pre = self._calculate_entropy(next_token_logits)
            metrics["entropy_history"].append(entropy_pre)
            
            # -- The Adaptive Trigger --
            if entropy_pre > entropy_threshold:
                metrics["triggers"] += 1
                start_time = time.time()

                # Grab the recent tail of generation to give some context to the query
                search_query = f"{query} {generated_text[-50:]}"
                context_doc = self.retriever.search(search_query)
                
                if context_doc:
                    print(f"\n--- Uncertainty Spike Detected ---")
                    print(f"Retrieving context for: '{search_query.strip()}'...")
                    
                    # Ghost prompt formulation
                    ghost_prompt = (
                        f"{current_input_str}"
                        f"\n# REFERENCE CODE:\n# {context_doc}\n# END REFERENCE\n"
                        f"{generated_text}"
                    )
                    ghost_ids = self.tokenizer(ghost_prompt, return_tensors="pt").input_ids.to(self.model.device)
                    
                    # Generate a short burst with the injected context
                    with torch.no_grad():
                        burst_output = self.model.generate(
                            ghost_ids, max_new_tokens=burst_tokens, do_sample=False, pad_token_id=self.tokenizer.eos_token_id
                        )
                    
                    new_ids = burst_output[0][ghost_ids.shape[1]:]
                    burst_text = self.tokenizer.decode(new_ids, skip_special_tokens=True)
                    
                    # See if confidence improved (calc delta)
                    full_temp = base_prompt + generated_text + burst_text
                    ids_temp = self.tokenizer(full_temp, return_tensors="pt").input_ids.to(self.model.device)
                    with torch.no_grad():
                        out_temp = self.model(ids_temp)
                        logits_post = out_temp.logits[:, -1, :]
                    
                    entropy_post = self._calculate_entropy(logits_post)
                    
                    end_time = time.time()
                    metrics["deltas"].append(entropy_pre - entropy_post)
                    metrics["latency"].append((end_time - start_time) * 1000)

                    # Keep the burst, discard the prompt
                    generated_text += burst_text
                    step += len(new_ids)
                    
                    # Resume normal sequence
                    current_input_str = base_prompt + generated_text
                    input_ids = self.tokenizer(current_input_str, return_tensors="pt").input_ids.to(self.model.device)
                    continue
            
            # Normal Step-by-Step
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            new_word = self.tokenizer.decode(next_token[0], skip_special_tokens=True)
            generated_text += new_word
            step += 1
            
            if next_token.item() == self.tokenizer.eos_token_id:
                break
        
        avg_delta = sum(metrics["deltas"])/len(metrics["deltas"]) if metrics["deltas"] else 0.0
        avg_latency = sum(metrics["latency"])/len(metrics["latency"]) if metrics["latency"] else 0.0

        return generated_text, {
            "triggers": metrics["triggers"], 
            "avg_delta_entropy": avg_delta, 
            "avg_intervention_latency_ms": avg_latency,
            "entropy_history": metrics["entropy_history"]
        }

### 4. Evaluation Formatting Utilities
LLM outputs can be pretty messy—markdown wrappers, loose comments, missing imports. These helper functions use some regex and indentation tricks to clean up the text into executable Python bodies. We need this to confidently pass the generated code into `exec()` for our HumanEval unit tests.

In [ ]:
def extract_imports(text: str) -> str:
    """Extract standard Python import statements from generated text."""
    imports = re.findall(r'^(?:from\s+\w+.*|import\s+\w+.*)', text, re.MULTILINE)
    return '\n'.join(sorted(list(set(imports)))) + '\n'

def extract_python_code(text: str) -> str:
    """Clean generated text to extract only executable Python code bodies."""
    # 1. Look for markdown code blocks
    match = re.search(r'```python\n(.*?)\n```', text, re.DOTALL)
    if match:
        code = match.group(1)
    else:
        # 2. Look for the start of function definitions
        match = re.search(r'(def\s+.*)', text, re.DOTALL)
        code = match.group(1) if match else text

    lines = code.split('\n')
    clean_lines = []
    
    # 3. Handle stop sequences like prints or comments
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("print(") or stripped.startswith("# Example") or stripped.startswith("### Expl"):
            break
        clean_lines.append(line)
        
    code_body = '\n'.join(clean_lines)
    
    # 4. Correct indentation (Critical Fix)
    # If the code block is an unindented body, indent appropriately for evaluation.
    if code_body and not code_body.startswith("def") and not code_body.startswith("    "):
        code_body = textwrap.indent(code_body, "    ")
        
    return code_body

def evaluate_code(code: str, test_cases: str) -> bool:
    """Execute code in a sandbox environment and return success boolean."""
    if not code: return False
    global_ns = {}
    # Redirect stdout to avoid polluting the terminal
    with contextlib.redirect_stdout(io.StringIO()):
        try:
            exec(code, global_ns)
            exec(test_cases, global_ns)
            return True
        except Exception:
            return False

### 5. Running the Pipeline
Since LLM generation can take a while, we split this into two parts:
1. **Generation:** Runs inference for both baseline and RAG models, continuously writing results to a `.jsonl` file. This acts as a checkpoint system in case the notebook crashes.
2. **Evaluation:** Reads the saved generations, cleans the code, and runs the actual HumanEval tests to compare performance and track our RAG metrics.

In [ ]:
def run_generation_phase(dataset, baseline_gen, rag_gen, output_file="experiments.jsonl", num_samples=10, entropy_threshold=2.0, burst_tokens=15):
    """Run model generation over a subset of the dataset and save results to disk."""
    print(f"Starting Generation Phase ({num_samples} samples)...")
    
    processed_ids = set()
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            for line in f:
                processed_ids.add(json.loads(line)['task_id'])

    for i, row in tqdm(dataset.head(num_samples).iterrows(), total=num_samples):
        task_id = row['task_id']
        if task_id in processed_ids: continue
        
        query = row['prompt']
        
        # 1. Baseline
        baseline_code, _ = baseline_gen.generate(query)
        
        # 2. Adaptive RAG
        rag_code, rag_metrics = rag_gen.generate_with_rag(query, entropy_threshold=entropy_threshold, burst_tokens=burst_tokens)
        
        entry = {
            "task_id": task_id,
            "prompt": query,
            "test_cases": row['test'],
            "baseline_output": baseline_code,
            "rag_output": rag_code,
            "rag_metrics": rag_metrics
        }
        
        with open(output_file, 'a') as f:
            f.write(json.dumps(entry) + "\n")
            
    print(f"Generation complete! Saved to {output_file}")

In [ ]:
def run_evaluation_phase(file_path="experiments.jsonl"):
    """Aggregate results from the generation phase and calculate performance metrics."""
    print(f"Evaluating results from {file_path}...")
    scores = {"baseline": 0, "rag": 0}
    total = 0
    rag_stats = {"triggers": [], "deltas": [], "latency": []}
    
    if not os.path.exists(file_path):
        print("Error: Results file not found.")
        return

    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            total += 1
            
            # Record RAG stats
            rag_stats["triggers"].append(data["rag_metrics"]["triggers"])
            if data["rag_metrics"]["avg_delta_entropy"] != 0:
                rag_stats["deltas"].append(data["rag_metrics"]["avg_delta_entropy"])

            if 'avg_intervention_latency_ms' in data["rag_metrics"]:
                rag_stats["latency"].append(data["rag_metrics"]["avg_intervention_latency_ms"])

            # Individual evaluation per model
            for m_key in ["baseline", "rag"]:
                raw_out = data[f"{m_key}_output"]
                
                imports = extract_imports(raw_out)
                body = extract_python_code(raw_out)
                full_test_code = f"{imports}\n{data['prompt']}\n{body}"
                
                if evaluate_code(full_test_code, data['test_cases']):
                    scores[m_key] += 1
    
    # Final Report
    print("\n" + "="*40)
    print(f"Results for {total} samples:")
    print("-" * 40)
    print(f"Baseline Pass Rate:     {scores['baseline']/total:.2%}")
    print(f"Adaptive RAG Pass Rate: {scores['rag']/total:.2%}")
    print("-" * 40)
    print("RAG System Metrics:")
    print(f"  - Avg Triggers per Sample:   {sum(rag_stats['triggers'])/total:.2f}")
    if rag_stats['deltas']:
        print(f"  - Avg Entropy Reduction (ΔH): {sum(rag_stats['deltas'])/len(rag_stats['deltas']):.4f}")
    if rag_stats['latency']:
        print(f"  - Avg Trigger Latency:        {sum(rag_stats['latency'])/len(rag_stats['latency']):.2f} ms")

### Run the whole thing
Now we glue it all together. We'll load up the **HumanEval** dataset along with our custom dataset for retrieval context. 

Feel free to tweak `ENTROPY_THRESHOLD` and `BURST_TOKENS` here to see how differently the RAG reacts to model uncertainty. lower entropy threshold = more frequent ghost prompts.

In [ ]:
# 1. Load HumanEval
print("Loading HumanEval dataset...")
dataset = load_dataset("openai_humaneval", split="test").to_pandas()

# 2. Load Retrival Context
# Edit this path depending on where your CSV is stored. Ask for the csv.
csv_path = "/kaggle/input/combined-problems-final-csv/combined_problems_final.csv"

if os.path.exists(csv_path):
    print(f"Reading corpus from: {csv_path}")
    df = pd.read_csv(csv_path)
    
    # Merge description and code to form the searchable documents
    corpus = (df['Description'].astype(str) + '\nCode:\n' + df[df.columns[2]].astype(str)).tolist()
    
    # --- Experiment Settings ---
    RRF_K = 60
    ENTROPY_THRESHOLD = 2.0
    BURST_TOKENS = 15
    NUM_SAMPLES = 15
    # ---------------------------
    
    # 3. Setup Models
    retriever_instance = ResearchRetriever(corpus, rrf_k=RRF_K)
    rag_gen_instance = AdaptiveRAGGenerator(retriever_instance)
    baseline_gen_instance = BaselineGenerator()

    # 4. Run the Pipeline!
    run_generation_phase(
        dataset, 
        baseline_gen_instance, 
        rag_gen_instance, 
        num_samples=NUM_SAMPLES, 
        entropy_threshold=ENTROPY_THRESHOLD, 
        burst_tokens=BURST_TOKENS
    )
    
    run_evaluation_phase()

else:
    print(f"Note: Could not find the corpus at {csv_path}. Please update the path.")

### Analyzing the Results
Once the JSONL trace is completely finished, we can break it down to see what actually happened.

- **`analyze_comparison_breakdown`:** Tells us if Ghost Prompting actively fixed a failing baseline case, or if it caused a regression.
- **`compare_generations_side_by_side`:** Prints a nice HTML table to directly compare the code output side by side.

In [ ]:
def analyze_comparison_breakdown(file_path="experiments.jsonl"):
    """
    Generate a comparative breakdown of Pass/Fail results for Baseline vs Adaptive RAG.
    Identifies specific instances where Adaptive RAG corrected common model failures.
    """
    print(f"Analyzing Pass/Fail breakdown from {file_path}...")

    results_list = []

    if not os.path.exists(file_path):
        print("ERROR: Result file not found.")
        return

    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            task_id = data['task_id']

            # Helper to evaluate specific outputs
            def evaluate_single_model(m_key):
                text = data[f"{m_key}_output"]
                imports = extract_imports(text)
                body = extract_python_code(text)
                full_code = f"{imports}\n{data['prompt']}\n{body}"
                return evaluate_code(full_code, data['test_cases'])

            baseline_passed = evaluate_single_model('baseline')
            rag_passed = evaluate_single_model('rag')

            # Determine comparative results
            if rag_passed and not baseline_passed:
                status = "Corrected by RAG"
            elif rag_passed and baseline_passed:
                status = "Baseline Success"
            elif not rag_passed and baseline_passed:
                status = "RAG Regression"
            else:
                status = "Critical Failure (Both)"

            # Aggregate RAG metrics
            metrics = data['rag_metrics']
            triggers = metrics['triggers']
            avg_delta = metrics['avg_delta_entropy']

            results_list.append({
                "Task ID": task_id,
                "Baseline": "PASSED" if baseline_passed else "FAILED",
                "Adaptive RAG": "PASSED" if rag_passed else "FAILED",
                "Triggers": triggers,
                "ΔH": f"{avg_delta:.3f}",
                "Status": status
            })

    df_results = pd.DataFrame(results_list)

    print("-" * 70)
    print("Instance Level Breakdown:")
    print("-" * 70)
    # Convert to markdown for clean console output
    print(df_results.to_markdown(index=False))

    corrected = df_results[df_results['Status'] == "Corrected by RAG"]

    if not corrected.empty:
        print("\n\n Instances where Adaptive RAG corrected Baseline failure:")
        print("-" * 40)
        print(f"Sample Problem ID: {corrected['Task ID'].iloc[0]}")
        print(f"RAG Triggers: {corrected['Triggers'].iloc[0]}")
        print(f"Entropy delta (ΔH): {corrected['ΔH'].iloc[0]}")
    else:
        print("\nNote: Baseline performance was highly accurate in this sample, or RAG failures matched baseline.")

    return df_results

# Execute analysis view
analyze_comparison_breakdown()


In [ ]:
def compare_generations_side_by_side(file_path="experiments.jsonl"):
    """
    Renders an HTML-styled table for side-by-side comparison of generated code.
    Visualizes the clean executable code used in evaluation.
    """
    if not os.path.exists(file_path):
        print(f" ERROR: Results file '{file_path}' not found.")
        return

    report_data = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)

            # Assemble Baseline executable
            imp_b = extract_imports(data['baseline_output'])
            body_b = extract_python_code(data['baseline_output'])
            code_b = f"<pre>{imp_b}{data['prompt']}{body_b}</pre>"

            # Assemble RAG executable
            imp_r = extract_imports(data['rag_output'])
            body_r = extract_python_code(data['rag_output'])
            code_r = f"<pre>{imp_r}{data['prompt']}{body_r}</pre>"

            # Pass/Fail Validation
            baseline_passed = evaluate_code(imp_b + data['prompt'] + body_b, data['test_cases'])
            rag_passed = evaluate_code(imp_r + data['prompt'] + body_r, data['test_cases'])

            # Determine Outcome
            if rag_passed and not baseline_passed:
                outcome = "Corrected Fault"
            elif rag_passed and baseline_passed:
                outcome = "Success (B & R)"
            elif not rag_passed and baseline_passed:
                outcome = "Regression"
            else:
                outcome = "Critical Fault"

            report_data.append({
                "Task ID": data['task_id'],
                "Outcome": outcome,
                "Triggers": data['rag_metrics']['triggers'],
                "ΔH": f"{data['rag_metrics']['avg_delta_entropy']:.3f}",
                "Baseline Code Block": code_b,
                "Adaptive RAG Block": code_r,
            })

    comparison_df = pd.DataFrame(report_data)

    # --- STYLE MAPPING ---
    def style_rows(row):
        styles = {
            "Corrected Fault": 'background-color: #d4edda; color: #155724;', # Soft Green
            "Success (B & R)": 'background-color: #f8f9fa;',               # Neutral
            "Regression": 'background-color: #f8d7da; color: #721c24;',     # Soft Red
            "Critical Fault": 'background-color: #f8d7da; color: #721c24;'
        }
        return [styles.get(row['Outcome'], '')] * len(row)

    styled_report = comparison_df.style.apply(style_rows, axis=1)

    # Set formatting for monospace code displays
    styled_report.set_properties(**{'font-family': 'monospace', 'font-size': '10pt'}, subset=['Baseline Code Block', 'Adaptive RAG Block'])

    print("VISUAL SIDE-BY-SIDE GENERATION COMPARISON")
    print("-" * 70)
    display(styled_report)


# Run stylized comparison
compare_generations_side_by_side()


In [ ]:
# Execution Cell: Extend and Re-evaluate Dataset

# 1. Targeted Sample Size for Extended Research
TOTAL_SAMPLES = 30 

# Ensure hyperparameters are defined in case this cell is run out of order
if 'ENTROPY_THRESHOLD' not in locals():
    ENTROPY_THRESHOLD = 2.0
if 'BURST_TOKENS' not in locals():
    BURST_TOKENS = 15

# 2. Resume Persistence Phase
# The generation script automatically bypasses already processed Task IDs.
print(f"Resuming generation phase. Processing until completion of {TOTAL_SAMPLES} unique samples.")

run_generation_phase(
    dataset, 
    baseline_gen_instance, 
    rag_gen_instance, 
    output_file="experiments.jsonl", 
    num_samples=TOTAL_SAMPLES,
    entropy_threshold=ENTROPY_THRESHOLD,
    burst_tokens=BURST_TOKENS
)

# 3. Final Evaluation Pass
print("\nExecuting comprehensive performance analysis on persistence file.")
run_evaluation_phase(file_path="experiments.jsonl")


In [ ]:
# Comprehensive Results Breakdown Summary
# (Redefinition of analysis function in case of isolated cell execution)

def generate_final_report(file_path="experiments.jsonl"):
    """
    Parses experiment logs to provide a final statistical and qualitative summary.
    """
    print(f"Analyzing final metrics from {file_path}...")

    final_results = []
    if not os.path.exists(file_path):
        print("Final Report Error: JSONL dataset missing.")
        return

    with open(file_path, 'r') as f:
        for line in f:
            entry = json.loads(line)
            
            # Helper evaluation logic
            def check_output(m_key):
                text = entry[f"{m_key}_output"]
                full_raw = f"{extract_imports(text)}\n{entry['prompt']}\n{extract_python_code(text)}"
                return evaluate_code(full_raw, entry['test_cases'])

            b_pass = check_output('baseline')
            r_pass = check_output('rag')

            status_label = "Stable"
            if r_pass and not b_pass: status_label = "RAG IMPROVED"
            elif not r_pass and b_pass: status_label = "RAG REGRESSION"
            elif not r_pass and not b_pass: status_label = "CHALLENGING CASE"

            final_results.append({
                "Task ID": entry['task_id'],
                "Baseline Pass": "YES" if b_pass else "NO",
                "RAG Pass": "YES" if r_pass else "NO",
                "Triggers": entry['rag_metrics']['triggers'],
                "Outcome": status_label
            })

    report_df = pd.DataFrame(final_results)
    print("-" * 50)
    print(report_df.to_markdown(index=False))
    
    counts = report_df['Outcome'].value_counts()
    print("\nSummary Statistics:")
    print(counts)

generate_final_report()
